# 00 – Business Understanding

First phase of **CRISP-DM (Business Understanding)**. This notebook describes the domain, the client's requirements and how they become data mining goals. It follows the three Business Understanding criteria of the grading rubric:

1. **Analysis of requirements with the end user** (Section 1)
2. **Definition of business goals** (Section 2)
3. **Translation of business goals into data mining goals** (Section 3)

Business Understanding is mostly text. The few code cells only **verify domain facts against the data** (league structure, schedule, points rule, playoff format), so every statement about the domain is backed by the database built with `src/load_db.py`.

| Phase | Notebook |
|---|---|
| 1. Business Understanding | `00_business_understanding.ipynb` (this notebook) |
| 2. Data Understanding | `01_eda.ipynb`, `02_statistical_analysis.ipynb` |
| 3. Data Preparation | `03_data_preparation.ipynb` |

In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../data/hockey.db")
q = lambda sql: pd.read_sql(sql, con)

## 1. Analysis of requirements with the end user

### 1.1 The game
- A game is played on an ice rink between two teams.
- Each team has six players on the ice at a time, including the goaltender.
- A game lasts one hour of gameplay (three periods of 20 minutes).
- The team that scores the most goals wins. A game tied after 60 minutes goes to a 5-minute overtime, which ends at the first goal. Until 2003/04, if nobody scored in overtime the game ended in a tie; since 2005/06 a shootout follows, so there are no ties. There was no 2004/05 season (lockout).
- Players who break the rules are sent to the penalty box. The minutes of each penalty are added to the player's **penalty minutes (PIM)**, the total over the season (target of requirement (c)).

| Penalty | Minutes (PIM) | Typical cause | Team short-handed? | Player returns in the game? |
|---|---|---|---|---|
| Minor | 2 | Tripping, hooking, slashing, holding, interference | Yes, until 2 min pass or the opponent scores | Yes |
| Double minor | 4 | High-sticking that draws blood | Yes, 4 min (a goal ends only the first half) | Yes |
| Major | 5 | Fighting, serious boarding or charging | Yes, the full 5 min, even if the opponent scores | Yes |
| Misconduct | 10 | Abuse of officials, unsportsmanlike behaviour | **No**, a teammate replaces him at once | Yes, after 10 min |
| Game misconduct | 10 | Serious misconduct; some majors (e.g. a third major in the same game) | No, by itself | **No**, ejected |
| Match penalty | Ejection; a teammate serves 5 min | Deliberate attempt to injure | Yes, 5 min | **No**, ejected (plus suspension) |

- When players of both teams are penalized at the same time (e.g. both fighters get 5 minutes), the penalties cancel out and neither team is short-handed. A penalized goaltender stays in the game and a teammate serves the penalty.
- So **PIM is not the same as time short-handed**: fights and misconducts add many minutes without giving the opponent a power play. A few misconducts can inflate a player's PIM, which makes target (c) noisy. The data only has the total PIM, not the type of each penalty.

### 1.2 The season
- Teams play a **regular season** inside a league divided into two conferences (Eastern and Western), each split into divisions. Teams accumulate points to reach the best possible standing.
- At the end of the regular season, the best teams qualify for the **playoffs**, where they play elimination series for the Stanley Cup.

### 1.3 League structure (last season in the data, 2009/10)

30 teams → 2 conferences of 15 → 3 geographic divisions of 5 per conference.

| Conference | Division | Teams (2009/10) |
|---|---|---|
| Eastern | Atlantic (AT) | New Jersey, Pittsburgh, Philadelphia, NY Rangers, NY Islanders |
| Eastern | Northeast (NE) | Boston, Buffalo, Ottawa, Montreal, Toronto |
| Eastern | Southeast (SE) | Washington, Carolina, Atlanta, Florida, Tampa Bay |
| Western | Central (CE) | Chicago, Detroit, St. Louis, Nashville, Columbus |
| Western | Northwest (NW) | Vancouver, Calgary, Minnesota, Colorado, Edmonton |
| Western | Pacific (PC) | San Jose, Phoenix, Los Angeles, Anaheim, Dallas |

Check in the data:

In [ ]:
structure = q('''
SELECT confID, divID, COUNT(*) AS n_teams, GROUP_CONCAT(name, ', ') AS teams
FROM Teams
WHERE lgID = 'NHL' AND year = 2009
GROUP BY confID, divID
ORDER BY confID, divID
''')
pd.set_option("display.max_colwidth", None)
structure

,confID,divID,n_teams,teams
0,EC,AT,5,"New Jersey Devils, New York Islanders, New York Rangers, Philadelphia Flyers, Pittsburgh Penguins"
1,EC,NE,5,"Boston Bruins, Buffalo Sabres, Montreal Canadiens, Ottawa Senators, Toronto Maple Leafs"
2,EC,SE,5,"Atlanta Thrashers, Carolina Hurricanes, Florida Panthers, Tampa Bay Lightning, Washington Capitals"
3,WC,CE,5,"Columbus Blue Jackets, Chicago Blackhawks, Detroit Red Wings, Nashville Predators, St. Louis Blues"
4,WC,NW,5,"Calgary Flames, Colorado Avalanche, Edmonton Oilers, Minnesota Wild, Vancouver Canucks"
5,WC,PC,5,"Anaheim Ducks, Dallas Stars, Los Angeles Kings, Phoenix Coyotes, San Jose Sharks"


### 1.4 Regular-season schedule (82 games per team)

Every team plays every other team, but not the same number of times:

| Opponent | Number of opponents | Games vs each | Total |
|---|---|---|---|
| Same division | 4 | 6 | 24 |
| Same conference, other division | 10 | 4 | 40 |
| Other conference | 15 | 1 or 2 (2 against 3 of them, by rotation) | 18 |
| | | | **82** |

- About 78% of the games (64 of 82) are inside the conference, and 24 are against division rivals, so **the strength of a team's division affects its points** (strength of schedule).
- This format holds in 2003/04, 2008/09 and 2009/10. From 2005/06 to 2007/08 teams played 8 games against each division rival (32) and only 10 inter-conference games, so each team did not meet 5 teams of the other conference. Before 2003/04 there were more inter-conference games (22–25 per team).

Check in `TeamVsTeam` for 2009:

In [ ]:
schedule = q('''
SELECT v.tmID, v.oppID,
       v.W + v.L + COALESCE(v.T, 0) + COALESCE(v.OTL, 0) AS games,
       CASE WHEN a.divID = b.divID THEN '1. Same division'
            WHEN a.confID = b.confID THEN '2. Same conference, other division'
            ELSE '3. Other conference' END AS relation
FROM TeamVsTeam v
JOIN Teams a ON a.year = v.year AND a.lgID = v.lgID AND a.tmID = v.tmID
JOIN Teams b ON b.year = v.year AND b.lgID = v.lgID AND b.tmID = v.oppID
WHERE v.lgID = 'NHL' AND v.year = 2009
''')
per_team = schedule.groupby(["tmID", "relation"]).agg(opponents=("oppID", "nunique"), games=("games", "sum"))
summary = per_team.groupby("relation").agg(
    opponents_per_team=("opponents", "mean"),
    games_per_team=("games", "mean"),
)
summary["games_vs_each"] = summary["games_per_team"] / summary["opponents_per_team"]
print(f"Games per team in 2009 (sum over opponents): {schedule.groupby('tmID')['games'].sum().unique()}")
summary.round(2)

Games per team in 2009 (sum over opponents): [82.]


,opponents_per_team,games_per_team,games_vs_each
relation,,,
1. Same division,4.0,24.0,6.0
"2. Same conference, other division",10.0,40.0,4.0
3. Other conference,15.0,18.0,1.2


In [ ]:
# Schedule split per season (games per team): same division / rest of conference / other conference
by_season = q('''
SELECT v.year, v.tmID, v.oppID,
       v.W + v.L + COALESCE(v.T, 0) + COALESCE(v.OTL, 0) AS games,
       CASE WHEN a.divID = b.divID THEN '1. Same division'
            WHEN a.confID = b.confID THEN '2. Same conference, other division'
            ELSE '3. Other conference' END AS relation
FROM TeamVsTeam v
JOIN Teams a ON a.year = v.year AND a.lgID = v.lgID AND a.tmID = v.tmID
JOIN Teams b ON b.year = v.year AND b.lgID = v.lgID AND b.tmID = v.oppID
WHERE v.lgID = 'NHL' AND v.year >= 1998
''')
n_teams = by_season.groupby("year")["tmID"].nunique()
split = by_season.groupby(["year", "relation"])["games"].sum().unstack().div(n_teams, axis=0)
split["opponents_met"] = by_season.groupby("year")["oppID"].count() / n_teams
split["possible_opponents"] = n_teams - 1
split.round(1)


### 1.5 Points and standings

| Result | Points |
|---|---|
| Win (regulation, overtime or shootout) | 2 |
| Overtime / shootout loss (OTL, since 1999/00) | 1 |
| Tie (only until 2003/04) | 1 |
| Regulation loss | 0 |

- $\text{Pts} = 2 \cdot W + T + \text{OTL}$. Since 2005/06 there are no ties: 5 minutes of overtime, then a shootout (there was no 2004/05 season, so this is the first season after 2003/04).
- Seasons have different lengths in 1982–2009 (80 games until 1991/92, 84 in 1992/93 and 1993/94, 48 in the 1994/95 lockout, 82 from 1995/96, and no 2004/05 season at all), so teams are compared with **points% = Pts / (2·G)**.
- Three standings exist at the same time, all sorted by points: division (5 teams; the `rank` column of `Teams`), conference (15 teams; decides the playoffs) and league (30 teams; the best team wins the Presidents' Trophy).
- Tie-break on equal points: more wins first (the official rule then uses head-to-head results and goal differential).
- **Conference standing:** the 3 division winners take places 1–3 of the conference, even with fewer points than other teams; the other 12 teams follow by points from 4th place.

Check of the points rule and of the season lengths:

In [ ]:
pts_check = q('''
SELECT COUNT(*) AS nhl_team_seasons_1982_2009,
       SUM(CASE WHEN Pts = 2 * W + COALESCE(T, 0) + COALESCE(OTL, 0) THEN 1 ELSE 0 END) AS rule_holds
FROM Teams
WHERE lgID = 'NHL' AND year >= 1982
''')
display(pts_check)

season_length = q('''
SELECT year, COUNT(*) AS teams, MIN(G) AS games_min, MAX(G) AS games_max
FROM Teams
WHERE lgID = 'NHL' AND year >= 1982
GROUP BY year
''')
season_length.groupby(["games_min", "games_max"])["year"].agg(["min", "max", "count"]).rename(
    columns={"min": "first_season", "max": "last_season", "count": "n_seasons"})

,nhl_team_seasons_1982_2009,rule_holds
0,690,690


,,first_season,last_season,n_seasons
games_min,games_max,,,
48,48,1994,1994,1
80,80,1982,1991,10
82,82,1995,2009,14
84,84,1992,1993,2


### 1.6 Playoffs
- The top 8 of each conference qualify (16 teams).
- Round 1 is played inside each conference: 1st vs 8th, 2nd vs 7th, 3rd vs 6th, 4th vs 5th. The following rounds are also inside the conference.
- The two conference champions play the Stanley Cup final. Every series is best of 7 (first to 4 wins).

Check: how many teams played the playoffs in 2009, per conference?

In [ ]:
q('''
SELECT t.confID, COUNT(*) AS playoff_teams
FROM TeamsPost p
JOIN Teams t ON t.year = p.year AND t.lgID = p.lgID AND t.tmID = p.tmID
WHERE p.lgID = 'NHL' AND p.year = 2009
GROUP BY t.confID
''')

,confID,playoff_teams
0,EC,8
1,WC,8


### 1.7 Requirements

Using the provided historical data, predict for the target season:

- **(a)** The final regular-season ranking of the teams.
- **(b)** The set of teams that will change coaches. Following the professor's clarification, a coaching change counts when it happens **during** the season, i.e. whether the team replaces its coach during the current season.
- **(c)** How many penalty minutes (PIM) each player will accumulate during the regular season. This requirement was announced later in the project. Because the database and the phase notebooks were already in place, it was added as a separate part of each phase (e.g. `sql/player_pim.sql`), without changing tasks (a) and (b).

### 1.8 End users and objectives

| Requirement | End users | How they would use the prediction |
|---|---|---|
| (a) Final ranking | Scouts, bettors, analysts, media | Scouting and roster planning, betting decisions, forecasting which teams reach the playoffs |
| (b) Coach change | Team owners and general managers, coaches and their agents, media, bettors | Owners/GMs: early warning that a change may be needed, so candidates and contracts can be prepared; coaches and agents: anticipate which jobs may open in the coaching market; media: coverage of coaches under pressure; bettors: "next coach to be fired" markets |
| (c) Penalty minutes | Coaching staff and general managers, fantasy hockey players, bettors | Coaches/GMs: plan the season (expected time short-handed, penalty-kill usage, which players need discipline work) and build a roster that balances physical and disciplined players when signing or trading; fantasy players: PIM is a scoring category in many fantasy leagues, so season forecasts guide the draft and trades; bettors: season-long player props (e.g. over/under on a player's total PIM), or a per-game baseline (predicted PIM / games) |

### 1.9 What each user needs from the output
- **(a)** A complete ranking of all teams. Users benefit from knowing which teams are likely to land near the playoff cut-off.
- **(b)** A list of teams predicted to change coach, with a probability or risk score per team.
- **(c)** A predicted number of regular-season PIM for every player expected to play in the target season, which can also be sorted to show the most penalized players (the ones that matter most to coaches, fantasy players and bettors).

### 1.10 Constraints and assumptions
- **Prediction time:** predictions are made the day before the first game of the new season. Only information available at that moment may be used.
- **Data:** only the historical data provided in the assignment is used.
- **Coaching change timing:** a change counts when it happens during the season (see 1.7).
- **Penalty minutes:** only regular-season PIM are predicted (playoff PIM are excluded), one value per player and season. A player traded during the season has one row per team in the data; his PIM are added up.

### 1.11 Expectations on quality
- **(a)** The predicted ranking should be close to the real final standings, and especially reliable at separating playoff teams from non-playoff teams.
- **(b)** The model should identify most of the teams that change coach without producing too many false alarms.
- **(c)** The predicted PIM should be close to the real values (error measured in minutes) and should rank the most penalized players correctly. Most players have few PIM and a few have many, so large errors on high-PIM players matter most.

These expectations become measurable success criteria in sections 2 and 3.

### 1.12 Open points

Points found while studying the domain that the later phases must handle:

| Open point | Found in | Status |
|---|---|---|
| **Division-winner rule:** the conference ranking must put the predicted division winners in places 1–3. It hardly changes who makes the playoffs, but it changes the order inside the top 8. | 1.5 | To apply when the ranking of task (a) is built (G1, section 3). |
| **Strength of schedule:** 24 of the 82 games are against division rivals, and 32 in 2005/06–2007/08. | 1.4 | Done: feature `div_rivals_pts_pct_rel` (mean relative points % of the division rivals), built in Data Understanding (`sql/playoff_and_sos.sql`). |
| **Traded players:** a player traded during the season has one row per team (`stint`). | 1.10 | Done in `sql/player_pim.sql`: PIM are added up over the stints, and the player's team is the one where he played the most games. |
| **Seasons without a previous season:** expansion teams (tasks a, b) and rookies (task c) have no season $t-1$. | 1.3, section 3 | Teams: done, flag `is_expansion_team` (Data Preparation, 5.2). Rookies: they need their own baseline in task (c). |
| **Rule changes between eras:** ties until 2003/04, OTL since 1999/00, different season lengths, different scoring levels. | 1.1, 1.5 | Done: targets are taken relative to the season (points %, `pts_pct_rel`) and the data is cut at 1982 (Data Preparation, section 3). |

### 1.13 Interaction with the client

The professor plays the role of the client. The requirements changed twice during the project:

| Topic | What the client said | Effect on the project |
|---|---|---|
| Requirement (b) | A coaching change counts when it happens **during** the season (a coach replaced mid-season), not between seasons. | Target (b) = the team has a second coach (`stint > 1`) in season $t$; the features describe the coach who started the season. |
| Requirement (c) | A third task was added: predict each player's regular-season PIM. | New unit of analysis (player-season instead of team-season), new end users (1.8) and goal G4. |

### 1.14 Inventory of resources

- **Data:** 22 CSV files from the assignment, loaded by `src/load_db.py` into one SQLite database (`data/hockey.db`), one table per file. They cover 1909–2009 (NHL from 1917/18 to 2009/10). The main tables are `Teams`, `TeamsPost` and `TeamVsTeam` (task a), `Coaches` and `AwardsCoaches` (task b), `Scoring` (task c), plus `Goalies` and `Master` (people's data). The `*SC` tables (Stanley Cup finals 1912–1925, several leagues) are not used.
- **Reusable SQL:** the queries that build the features are kept in `sql/`, one file per topic (coaches, teams, goalies, PIM).
- **Software:** Python with SQLite, pandas and NumPy for the data, SciPy and statsmodels for the statistical tests, scikit-learn for the models, and matplotlib and seaborn for the figures.
- **Domain knowledge:** the NHL rules (section 1), each checked against the data when the data allows it.

### 1.15 Risks and contingencies

| Risk | Why it matters | Contingency |
|---|---|---|
| Few positives in task (b) (about 19% of team-seasons) | A model can reach 81% accuracy by never predicting a change | PR-AUC instead of accuracy (section 3); class weights; the alarm threshold is chosen by the cost of each error (section 2) |
| Data leakage | Information from season $t$ (e.g. `TeamsPost`, the mid-season coach, same-season statistics) would make the results look better than they can be in practice | Features only up to $t-1$; walk-forward evaluation (section 3); a timeliness check of every feature (Data Preparation, 2.6) |
| Rules and calendar change between eras | Points, ties and season lengths are not comparable across seasons | Targets relative to the season; data cut at 1982 |
| Few seasons to evaluate (9 seasons × 30 teams) | Metrics change a lot from season to season | Report the value of every season, not only the mean, and compare with the baseline on the same seasons |
| PIM is noisy and skewed | A few fights or misconducts can add many minutes; most players have few PIM and a few have many | MAE and Spearman (section 3); special attention to the high-PIM players |
| Missing history (expansion teams, rookies, first-time coaches) | Features of $t-1$ are missing for these rows | Indicator flags and imputation fitted only on past seasons (Data Preparation, section 5) |
| The data ends in 2009/10 | Later rule changes (e.g. 3-on-3 overtime from 2015/16) are not in the data | The predictions assume the rules of the last seasons in the data |
| Requirements change during the project | New tasks can force rework | Shared database and one notebook per phase (see 1.13 and requirement (c)) |

### 1.16 Glossary

| Term | Meaning |
|---|---|
| Season $t$ / `year` | A season is identified by the year it starts (`year = 2009` is the 2009/10 season) |
| `lgID`, `confID`, `divID` | League (NHL, WHA, …), conference and division of a team |
| `tmID`, `franchID` | Team code of a season and franchise code; the franchise stays the same when a team moves or changes name |
| W, L, T, OTL | Wins, regulation losses, ties (until 2003/04) and overtime/shootout losses (since 1999/00) |
| Pts, points % | Standings points ($2 \cdot W + T + OTL$) and Pts / (2·G), the share of the maximum possible points |
| `pts_pct_rel` | Points % minus the average of the season, so that seasons can be compared |
| PIM | Penalty minutes: the total minutes of all the penalties of a player (or team) in a season |
| Power play / penalty kill | Playing with one more / one fewer skater because of a penalty |
| `stint` | The order of a player's or coach's spells with different teams in the same season (1, 2, 3) |
| Playoffs | The elimination series after the regular season (top 8 of each conference) |
| Walk-forward | Evaluation where the model is trained on the seasons before $t$ and tested on season $t$, for each $t$ in turn |


## 2. Definition of business goals

Each requirement of 1.7 becomes a business goal, written from the point of view of its end users (1.8), with a **business success criterion**: what the users must get for the prediction to be worth more than what they could do without data mining. Without a model, every user would simply assume that **this season will look like the last one**, so each criterion is set against that *same as last season* baseline, measured in the data below on the seasons 2000–2009.

| # | Business goal | Main end users | Business success criterion |
|---|---|---|---|
| G1 | I want to know, before the season starts, how the teams will finish the regular season. | Scouts, bettors, analysts, media | The predicted standings are closer to the final standings than last season's standings are. |
| G2 | I want to know, before the season starts, which teams will qualify for the playoffs. | Bettors, analysts, media | On average, more of the 16 playoff teams are identified than the ~11 that last season's standings identify. |
| G3 | I want to know, before the season starts, which teams are at risk of replacing their coach during the season. | Owners and GMs, coaches and agents, media, bettors | A short list of high-risk teams that contains most of the 5–6 teams per season that change coach, with far fewer false alarms than picking teams at random (about 1 team in 5 changes coach). |
| G4 | I want to know, before the season starts, how many penalty minutes each player will accumulate. | Coaching staff and GMs, fantasy players, bettors | Smaller errors than assuming each player repeats last season's PIM, and the most penalized players of the season correctly identified. |

- **G1 and G2 come from the same requirement (a).** The playoff list is read from the predicted standings (top 8 of each conference, see 1.6), so the two answers can never contradict each other.
- **Not every error costs the same:**
  - G2: mistakes near the cut-off (8th vs 9th place) are the ones that change a bettor's decision.
  - G3: a missed change leaves a team unprepared, while a false alarm only costs attention, so finding the changes matters more than avoiding every false alarm.
  - G4: errors on the high-PIM players matter most (see 1.11).

Baselines measured in the data (seasons 2000–2009):


In [2]:
# "Same as last season" baselines over the evaluation seasons 2000–2009 (there is no 2004/05 season)
import numpy as np
from scipy.stats import spearmanr

EVAL = [y for y in range(2000, 2010) if y != 2004]

def add_prev(df, key, cols):
    """Join the values of the previous season that exists (2003 for 2005)."""
    years = sorted(df["year"].unique())
    prev_year = dict(zip(years[1:], years[:-1]))
    df = df.assign(prev_year=df["year"].map(prev_year))
    prev = df[["year", key] + cols].rename(columns={"year": "prev_year", **{c: c + "_prev" for c in cols}})
    return df.merge(prev, on=["prev_year", key])  # rows without a previous season (expansion teams, rookies) drop out

# (a) Ranking and playoffs: predict this season's points % with last season's
teams = q('''
SELECT t.year, t.tmID, t.franchID, t.confID, 1.0 * t.Pts / (2 * t.G) AS pts_pct,
       CASE WHEN p.tmID IS NULL THEN 0 ELSE 1 END AS playoffs
FROM Teams t
LEFT JOIN TeamsPost p ON p.year = t.year AND p.lgID = t.lgID AND p.tmID = t.tmID
WHERE t.lgID = 'NHL' AND t.year >= 1998
''')
teams["pts_pct_rel"] = teams["pts_pct"] - teams.groupby("year")["pts_pct"].transform("mean")
teams = add_prev(teams, "franchID", ["pts_pct_rel"])
rows = []
for y in EVAL:
    d = teams[teams["year"] == y]
    top8 = d.sort_values("pts_pct_rel_prev", ascending=False).groupby("confID").head(8)
    rows.append({"season": y,
                 "RMSE pts_pct_rel": np.sqrt(((d["pts_pct_rel"] - d["pts_pct_rel_prev"]) ** 2).mean()),
                 "Spearman ranking": spearmanr(d["pts_pct_rel"], d["pts_pct_rel_prev"]).statistic,
                 "playoff teams hit (of 16)": int(top8["playoffs"].sum())})
baseline_a = pd.DataFrame(rows).set_index("season")

# (b) Coach change during the season: share of team-seasons with a change (= PR-AUC of a random ranking)
coach = q('''
SELECT year, tmID, MAX(stint) > 1 AS changed
FROM Coaches WHERE lgID = 'NHL' AND year BETWEEN 2000 AND 2009
GROUP BY year, tmID
''')

# (c) Penalty minutes: predict this season's PIM with last season's (stints added up)
pim = q('''
SELECT playerID, year, SUM(PIM) AS pim
FROM Scoring WHERE lgID = 'NHL' AND year >= 1998
GROUP BY playerID, year
''').dropna()
pim = add_prev(pim, "playerID", ["pim"])
pim = pim[pim["year"].isin(EVAL)]

display(baseline_a.round(3))
summary = pd.DataFrame({
    "Task": ["(a) points %", "(a) ranking", "(a) playoffs", "(b) coach change", "(c) PIM", "(c) PIM"],
    "Baseline": ["same as last season"] * 3 + ["random ranking", "same as last season", "same as last season"],
    "Metric": ["RMSE of pts_pct_rel", "Spearman", "playoff teams hit (of 16)", "PR-AUC (= share of positives)",
               "MAE (minutes)", "Spearman"],
    "Value 2000–2009": [baseline_a["RMSE pts_pct_rel"].mean(), baseline_a["Spearman ranking"].mean(),
                        baseline_a["playoff teams hit (of 16)"].mean(), coach["changed"].mean(),
                        (pim["pim"] - pim["pim_prev"]).abs().mean(),
                        spearmanr(pim["pim"], pim["pim_prev"]).statistic],
})
summary.round(3)

,RMSE pts_pct_rel,Spearman ranking,playoff teams hit (of 16)
season,,,
2000,0.071,0.791,14
2001,0.087,0.536,11
2002,0.091,0.494,10
2003,0.084,0.562,12
2005,0.094,0.457,11
2006,0.100,0.558,11
2007,0.088,0.485,11
2008,0.065,0.554,11
2009,0.082,0.474,10


,Task,Baseline,Metric,Value 2000–2009
0,(a) points %,same as last season,RMSE of pts_pct_rel,0.085
1,(a) ranking,same as last season,Spearman,0.546
2,(a) playoffs,same as last season,playoff teams hit (of 16),11.222
3,(b) coach change,random ranking,PR-AUC (= share of positives),0.193
4,(c) PIM,same as last season,MAE (minutes),19.522
5,(c) PIM,same as last season,Spearman,0.709


## 3. Translation of business goals into data mining goals

**Setup common to all goals:**
- **One row per unit and season $t$.** Features use only information known the day before season $t$: statistics up to season $t-1$ and the coaches' careers (see 1.10).
- **Evaluation:** walk-forward over $t$ = 2000–2009, without 2004: train on the seasons before $t$, predict season $t$, and average the metric over the 9 seasons. This is the same scheme used in Data Preparation (`03_data_preparation.ipynb`, section 5.3).
- **Success criterion:** beat the *same as last season* baseline of section 2, on the same seasons and the same metric.

| Goal | Data mining goal | Type | Unit and target | Metric | Success criterion (mean 2000–2009) |
|---|---|---|---|---|---|
| G1 | Predict each team's points % relative to the season average, then sort the teams within division, conference and league (division winners in places 1–3 of the conference, see 1.5) | Regression, then ranking | Team-season; `pts_pct_rel` = Pts / (2·G) − mean of the season | RMSE of `pts_pct_rel`; Spearman between predicted and real ranking | RMSE < 0.085 and Spearman > 0.55 |
| G2 | Take the top 8 of each conference of the predicted ranking of G1 | Binary decision derived from G1 (no separate model) | Team-season; playoffs yes / no | Playoff teams hit, out of 16 | More than 11.2 of 16 |
| G3 | Predict the probability that the team changes coach during season $t$ and rank the teams by it | Binary classification, imbalanced (~19% positives) | Team-season; 1 if a second coach (`stint > 1`) takes over during $t$ | PR-AUC (main), ROC-AUC, recall among the 8 highest-risk teams | PR-AUC clearly above 0.19 (random ranking) and ROC-AUC > 0.5 |
| G4 | Predict each player's regular-season PIM in season $t$ | Regression (count target, skewed) | Player-season; PIM added up over all the player's stints (see 1.10) | MAE in minutes; Spearman | MAE < 19.5 minutes and Spearman > 0.71 |

**Why these metrics:**
- **G1:** the target is relative to the season average, so seasons with different scoring levels can be compared. RMSE punishes large misses, and Spearman measures what the users actually see, which is the order of the teams.
- **G2:** the number of playoff teams hit is exactly the business criterion. Since 16 teams are always predicted, precision and recall are the same number.
- **G3:** with 19% positives, predicting "no team changes coach" already gives 81% accuracy and is useless, so accuracy is not used. PR-AUC focuses on the positive class, and its value for a random ranking is the share of positives (0.19), which makes a clear reference.
- **G4:** MAE is in minutes, so users can read it directly ("off by *x* minutes per player"). Spearman checks that the most penalized players are ranked at the top. The baseline only covers players with a previous NHL season, so rookies need their own reference (for example, the median PIM of past rookies).

In [ ]:
con.close()